<!-- 🎯 Objective:
Build a multi-agent customer support system that can classify, delegate, and resolve user queries.

💼 Problem Statement
A company wants to automate its support system using AI agents.

The system should:

Understand customer queries
Route them to the correct department
Provide responses
👥 Agents to Build
Classifier Agent
Identifies query type (Billing / Technical / General)
Billing Agent
Handles payment/refund queries
Technical Support Agent
Handles app/software issues
Response Agent
Combines outputs into a final response
⚙️ Tasks to Implement
Multi-agent architecture (at least 3 agents mandatory)
Task delegation logic
Basic agent communication (message passing / function calls)
💡 Sample Inputs
“I was charged twice for my subscription”
“The app crashes when I open it”
“What are your pricing plans?”
🧪 Expected Output
Correct classification of query
Delegation to appropriate agent
Final structured response -->

In [ ]:
# Workflow Idea

# We create a graph where:

# Classifier Node → detects query type
# Route to correct node
# Specialized agent solves query
# Response node formats final output

In [ ]:
#Architecture Digram
# START
#   │
#   ▼
# [Classifier Node]
#   │
#   ├── if Billing   ──► [Billing Node]
#   │
#   ├── if Technical ──► [Technical Node]
#   │
#   └── if General   ──► [General Node]
#                             │
#                             ▼
#                     [Response Node]
#                             │
#                             ▼
#                            END

In [2]:
import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# load env
load_dotenv()

# LLM using NVIDIA API
llm = ChatOpenAI(
    model="meta/llama3-70b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0
)

# ---------------- STATE ---------------- #

class SupportState(TypedDict):
    query: str
    category: str
    answer: str

# ---------------- AGENTS ---------------- #

def classifier_agent(state: SupportState):

    prompt = f"""
    Classify query into one word only:
    Billing
    Technical
    General

    Query:
    {state['query']}
    """

    response = llm.invoke(prompt).content

    state["category"] = response.strip()

    return state


def billing_agent(state: SupportState):

    prompt = f"""
    Provide professional billing support response.

    Query:
    {state['query']}
    """

    response = llm.invoke(prompt).content

    state["answer"] = response

    return state


def technical_agent(state: SupportState):

    prompt = f"""
    Provide technical troubleshooting response.

    Query:
    {state['query']}
    """

    response = llm.invoke(prompt).content

    state["answer"] = response

    return state


def general_agent(state: SupportState):

    prompt = f"""
    Answer general customer question.

    Query:
    {state['query']}
    """

    response = llm.invoke(prompt).content

    state["answer"] = response

    return state


def response_agent(state: SupportState):

    state["answer"] = f"""
Category: {state['category']}

Response:
{state['answer']}
"""

    return state


# ---------------- ROUTER ---------------- #

def route_query(state: SupportState):

    if "billing" in state["category"].lower():
        return "billing"

    elif "technical" in state["category"].lower():
        return "technical"

    else:
        return "general"


# ---------------- GRAPH ---------------- #

builder = StateGraph(SupportState)

builder.add_node("classifier", classifier_agent)

builder.add_node("billing", billing_agent)

builder.add_node("technical", technical_agent)

builder.add_node("general", general_agent)

builder.add_node("response", response_agent)

# edges
builder.set_entry_point("classifier")

builder.add_conditional_edges(
    "classifier",
    route_query,
    {
        "billing": "billing",
        "technical": "technical",
        "general": "general"
    }
)

builder.add_edge("billing", "response")
builder.add_edge("technical", "response")
builder.add_edge("general", "response")

builder.add_edge("response", END)

# compile graph
graph = builder.compile()

# ---------------- TEST ---------------- #

queries = [
    "I was charged twice for my subscription",
    "The app crashes when I open it",
    "What are your pricing plans?"
]

for q in queries:

    result = graph.invoke({
        "query": q
    })

    print("\n====================")
    print(result["answer"])



Category: Billing

Response:
Dear [Customer's Name],

Thank you for reaching out to us regarding the issue with your subscription billing. We apologize for the inconvenience this has caused and are committed to resolving this matter as quickly as possible.

We have reviewed your account and found that there was an error on our part that resulted in a duplicate charge. Please be assured that we are taking immediate action to correct this issue.

To rectify the situation, we will be issuing a refund for the duplicate charge in the amount of [amount] within the next 3-5 business days. You will receive an email notification once the refund has been processed.

In the meantime, we have also taken steps to ensure that your subscription is updated to reflect the correct billing information, and you will not be charged again until your next scheduled billing cycle.

If you have any further questions or concerns, please do not hesitate to contact us. Your satisfaction is our top priority, and